# Train + evaluate (TPU)

Runs the full pipeline end to end: generate both datasets, fine-tune with LoRA, then evaluate direction accuracy. Works in Colab, Kaggle, or RunPod (or any GPU box) - see `README.md`'s "Pipeline"/"Running on RunPod" sections for the full explanation of each step.

**Secrets/env vars** - Colab/Kaggle: set these via the key icon in the left sidebar (**Secrets**), and grant this notebook access when prompted. RunPod or any plain GPU box (no notebook secrets vault): set the same names as real environment variables before starting Jupyter (`export NAME=value`, or your pod's env-var config) - `get_secret()` in every script below falls back to `os.environ` automatically outside Colab/Kaggle, no code change needed.

| Name | Required | Used by |
|---|---|---|
| `HF_TOKEN` | yes | train_model.py (pushes the fine-tuned model) |
| `HF_USER` | yes | train_model.py, evaluate_model.py |
| `GEMINI_API_KEY` | yes | generate_real_dataset.py |
| `MODEL_CHOICE` | optional | override the base model family for this run only |
| `MODEL_VERSION` | optional | override the pushed version to reload/compare |
| `SKIP_BASE_MODEL_EVAL` | optional | set to `1` to skip evaluate_model.py's base-model comparison pass |

**Runtime:** needs a TPU v5e-1 (plain transformers/peft/trl, no unsloth/bitsandbytes). Pick this notebook for exactly one accelerator per training run - see "GPU vs TPU" in the README for why they're separate implementations.

Run all cells in order (Runtime -> Run all).

In [ ]:
import os
if os.path.isdir("financial-sentiment-model"):
    !rm -rf financial-sentiment-model
!git clone --depth 1 https://github.com/GiulianoAparecido/financial-sentiment-model.git
%cd financial-sentiment-model

## Steps 1-3: generate the datasets (hardware-agnostic, same either way)

In [ ]:
!pip install -q yfinance httpx feedparser google-genai pandas

In [ ]:
%run generate_synthetic_dataset.py

In [ ]:
%run generate_real_dataset.py

## Step 4: train (TPU)

In [ ]:
%run notebooks/train/tpu/train_model.py

## Step 5: evaluate (reuses the model/tokenizer already in memory from step 4)

In [ ]:
%run notebooks/train/tpu/evaluate_model.py